In [ ]:
%run ./1_config


In [ ]:

from pyspark.sql import functions as F

class CleanupHelper:
    def __init__(self):
        self.c = conf
        self.cutoff = F.date_sub(F.current_date(), self.c.lookback_days)

    def purge(self, table):
        fqn = self.c.table_fqn(table)
        if not spark.catalog.tableExists(fqn):
            print(f"skip {fqn}")
            return
        before = spark.table(fqn).count()
        spark.sql(f"DELETE FROM {fqn} WHERE event_date < date_sub(current_date(), {self.c.lookback_days})")
        after = spark.table(fqn).count()
        print(f"✅ {table}: {before} -> {after} rows")

    def run(self):
        for t in [self.c.bronze_events, self.c.silver_events_enriched]:
            self.purge(t)

CleanupHelper().run()

